# LSTM - Grid Search & Random Search

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import time
import itertools
import random
from collections import Counter
from data_loader import load_clinc

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = True
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda
1525 / 620 / 1100, 151 classes (sample)


In [2]:
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        counts = Counter()
        for t in texts:
            counts.update(t.lower().split())
        idx = 2
        for word, cnt in counts.items():
            if cnt >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, text):
        return [self.word2idx.get(w, 1) for w in text.lower().split()]

    def __len__(self):
        return len(self.word2idx)


vocab = Vocabulary(min_freq=2)
vocab.build_vocab(train_texts)
vocab_size = len(vocab)
print(f"Vocab: {vocab_size}")

Vocab: 876


In [3]:
class TextSequenceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.LongTensor(self.vocab.encode(self.texts[idx])), torch.LongTensor([self.labels[idx]])


def collate_batch(batch):
    seqs, labels = zip(*batch)
    return pad_sequence(seqs, batch_first=True, padding_value=0), torch.LongTensor([len(s) for s in seqs]), torch.cat(labels)


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)

    def forward(self, sequences, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.embedding(sequences), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        return self.fc(self.dropout(hidden))


train_ds = TextSequenceDataset(train_texts, train_labels, vocab)
val_ds = TextSequenceDataset(val_texts, val_labels, vocab)

In [4]:
PATIENCE = 5
MAX_EPOCHS = 30


def evaluate_hyperparams(params):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, collate_fn=collate_batch)
    vl_loader = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_batch)

    model = LSTMClassifier(
        vocab_size, params['embedding_dim'], params['hidden_dim'], num_classes,
        num_layers=params['num_layers'], dropout=params['dropout'], bidirectional=True
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for seqs, lengths, labels in tr_loader:
            seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(seqs, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for seqs, lengths, labels in vl_loader:
                all_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)
        scheduler.step(vl_f1)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    return best_f1, epoch + 1

## Grid Search

In [5]:
grid = {
    'embedding_dim': [64, 128, 256],
    'hidden_dim': [128, 256, 512],
    'num_layers': [1, 2],
    'dropout': [0.2, 0.3],
    'learning_rate': [0.0005, 0.001],
    'batch_size': [64]
}

keys = list(grid.keys())
combinations = list(itertools.product(*grid.values()))
print(f"Total combinations: {len(combinations)}")

grid_results = []
best_f1, best_params = 0, None
start_time = time.time()

for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    f1, epochs = evaluate_hyperparams(params)
    grid_results.append({"params": params, "val_f1": round(f1, 4), "epochs": epochs})

    if f1 > best_f1:
        best_f1, best_params = f1, params

    print(f"{i+1:3d}/{len(combinations)}  f1={f1:.4f}  emb={params['embedding_dim']} hid={params['hidden_dim']} "
          f"layers={params['num_layers']} drop={params['dropout']} lr={params['learning_rate']}")

grid_time = time.time() - start_time
print(f"\nGrid Search done. Best F1: {best_f1:.4f}, Time: {grid_time:.1f}s")
print(f"Best params: {best_params}")

Total combinations: 72
  1/72  f1=0.4244  emb=64 hid=128 layers=1 drop=0.2 lr=0.0005
  2/72  f1=0.4511  emb=64 hid=128 layers=1 drop=0.2 lr=0.001
  3/72  f1=0.4152  emb=64 hid=128 layers=1 drop=0.3 lr=0.0005
  4/72  f1=0.4527  emb=64 hid=128 layers=1 drop=0.3 lr=0.001
  5/72  f1=0.4476  emb=64 hid=128 layers=2 drop=0.2 lr=0.0005
  6/72  f1=0.4350  emb=64 hid=128 layers=2 drop=0.2 lr=0.001
  7/72  f1=0.4667  emb=64 hid=128 layers=2 drop=0.3 lr=0.0005
  8/72  f1=0.4501  emb=64 hid=128 layers=2 drop=0.3 lr=0.001
  9/72  f1=0.4325  emb=64 hid=256 layers=1 drop=0.2 lr=0.0005
 10/72  f1=0.4375  emb=64 hid=256 layers=1 drop=0.2 lr=0.001
 11/72  f1=0.4260  emb=64 hid=256 layers=1 drop=0.3 lr=0.0005
 12/72  f1=0.4163  emb=64 hid=256 layers=1 drop=0.3 lr=0.001
 13/72  f1=0.4160  emb=64 hid=256 layers=2 drop=0.2 lr=0.0005
 14/72  f1=0.4504  emb=64 hid=256 layers=2 drop=0.2 lr=0.001
 15/72  f1=0.4258  emb=64 hid=256 layers=2 drop=0.3 lr=0.0005
 16/72  f1=0.4517  emb=64 hid=256 layers=2 drop=0.3 lr

## Random Search

In [6]:
random_space = {
    'embedding_dim': [64, 128, 256],
    'hidden_dim': [128, 256, 512],
    'num_layers': [1, 2, 3],
    'dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005],
    'batch_size': [32, 64, 128]
}

N_RANDOM_ITER = len(combinations)  # same budget as grid search

random.seed(42)
random_results = []
best_f1_rand, best_params_rand = 0, None
start_time = time.time()

for i in range(N_RANDOM_ITER):
    params = {k: random.choice(v) for k, v in random_space.items()}
    f1, epochs = evaluate_hyperparams(params)
    random_results.append({"params": params, "val_f1": round(f1, 4), "epochs": epochs})

    if f1 > best_f1_rand:
        best_f1_rand, best_params_rand = f1, params

    print(f"{i+1:3d}/{N_RANDOM_ITER}  f1={f1:.4f}  emb={params['embedding_dim']} hid={params['hidden_dim']} "
          f"layers={params['num_layers']} drop={params['dropout']} lr={params['learning_rate']} bs={params['batch_size']}")

random_time = time.time() - start_time
print(f"\nRandom Search done. Best F1: {best_f1_rand:.4f}, Time: {random_time:.1f}s")
print(f"Best params: {best_params_rand}")

  1/72  f1=0.5550  emb=256 hid=128 layers=1 drop=0.3 lr=0.0005 bs=32
  2/72  f1=0.0935  emb=64 hid=512 layers=1 drop=0.5 lr=0.0001 bs=128
  3/72  f1=0.4894  emb=128 hid=128 layers=1 drop=0.1 lr=0.0005 bs=32
  4/72  f1=0.5014  emb=256 hid=512 layers=1 drop=0.5 lr=0.0005 bs=128
  5/72  f1=0.4588  emb=256 hid=512 layers=3 drop=0.4 lr=0.0005 bs=64
  6/72  f1=0.5070  emb=256 hid=256 layers=1 drop=0.2 lr=0.005 bs=64
  7/72  f1=0.2917  emb=128 hid=128 layers=1 drop=0.3 lr=0.0001 bs=32
  8/72  f1=0.4924  emb=128 hid=128 layers=2 drop=0.3 lr=0.001 bs=32
  9/72  f1=0.4492  emb=256 hid=256 layers=3 drop=0.1 lr=0.005 bs=32
 10/72  f1=0.5174  emb=256 hid=256 layers=3 drop=0.5 lr=0.001 bs=128
 11/72  f1=0.4308  emb=64 hid=512 layers=1 drop=0.1 lr=0.0005 bs=64
 12/72  f1=0.4585  emb=64 hid=128 layers=1 drop=0.4 lr=0.001 bs=64
 13/72  f1=0.5465  emb=256 hid=256 layers=1 drop=0.3 lr=0.001 bs=32
 14/72  f1=0.4806  emb=256 hid=256 layers=3 drop=0.1 lr=0.0005 bs=128
 15/72  f1=0.5286  emb=256 hid=128 laye

## Test Evaluation

In [7]:
def train_and_test(params, label):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, collate_fn=collate_batch)
    vl_loader = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_batch)
    te_loader = DataLoader(TextSequenceDataset(test_texts, test_labels, vocab),
                           batch_size=params['batch_size'], shuffle=False, collate_fn=collate_batch)

    model = LSTMClassifier(
        vocab_size, params['embedding_dim'], params['hidden_dim'], num_classes,
        num_layers=params['num_layers'], dropout=params['dropout'], bidirectional=True
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_f1, best_state, patience_counter = 0, None, 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for seqs, lengths, labels in tr_loader:
            seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(seqs, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        vl_preds = []
        with torch.no_grad():
            for seqs, lengths, labels in vl_loader:
                vl_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
        scheduler.step(vl_f1)
        if vl_f1 > best_f1:
            best_f1, best_state, patience_counter = vl_f1, model.state_dict().copy(), 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    te_preds = []
    with torch.no_grad():
        for seqs, lengths, labels in te_loader:
            te_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

    acc = accuracy_score(test_labels, te_preds)
    p = precision_score(test_labels, te_preds, average='macro', zero_division=0)
    r = recall_score(test_labels, te_preds, average='macro', zero_division=0)
    f1_m = f1_score(test_labels, te_preds, average='macro', zero_division=0)
    f1_w = f1_score(test_labels, te_preds, average='weighted', zero_division=0)

    print(f"{label}:")
    print(f"    Accuracy:   {acc:.4f}")
    print(f"    Precision:  {p:.4f}")
    print(f"    Recall:     {r:.4f}")
    print(f"    F1 macro:   {f1_m:.4f}")
    print(f"    F1 weighted:{f1_w:.4f}")

    return {
        "accuracy": round(acc, 4), "macro_precision": round(p, 4),
        "macro_recall": round(r, 4), "macro_f1": round(f1_m, 4), "weighted_f1": round(f1_w, 4)
    }


grid_metrics = train_and_test(best_params, "Grid Search")
print()
rand_metrics = train_and_test(best_params_rand, "Random Search")

Grid Search:
    Accuracy:   0.5291
    Precision:  0.5880
    Recall:     0.5674
    F1 macro:   0.5396
    F1 weighted:0.5211

Random Search:
    Accuracy:   0.5482
    Precision:  0.5871
    Recall:     0.5756
    F1 macro:   0.5497
    F1 weighted:0.5397


## Save Results

In [8]:
gs_out = {
    "model_type": "LSTM",
    "optimization": "grid_search",
    "best_hyperparameters": best_params,
    "total_evaluations": len(combinations),
    "search_time_seconds": round(grid_time, 2),
    "test_metrics": grid_metrics,
    "all_results": grid_results
}
with open('results/results_lstm_grid.json', 'w') as f:
    json.dump(gs_out, f, indent=4)

rs_out = {
    "model_type": "LSTM",
    "optimization": "random_search",
    "best_hyperparameters": best_params_rand,
    "total_evaluations": N_RANDOM_ITER,
    "search_time_seconds": round(random_time, 2),
    "test_metrics": rand_metrics,
    "all_results": random_results
}
with open('results/results_lstm_random.json', 'w') as f:
    json.dump(rs_out, f, indent=4)

print("Saved: results/results_lstm_grid.json, results/results_lstm_random.json")

Saved: results/results_lstm_grid.json, results/results_lstm_random.json
